# CRIS — Переобучение моделей v2

Обучает три модели на **усреднённых 200-dim KDE-векторах** — совместимо с inference-пайплайном.

Выходные файлы:
- `rf_optimized_model_v2.pkl`
- `catboost_substance_v2.cbm`
- `extra_trees_v2.pkl`

**Структура ожидаемого zip:**
```
<label>/
  <iteration>/
    kde_array_*.csv   ← один файл на ион
```
Каждый CSV содержит колонку `kde_values` (200 значений).

In [ ]:
# ── Зависимости ───────────────────────────────────────────────────────────────
!pip install catboost flaml[automl] scikit-learn joblib pandas numpy -q

In [ ]:
# ── Загрузка zip-архива ───────────────────────────────────────────────────────
from google.colab import files
uploaded = files.upload()
ZIP_PATH = list(uploaded.keys())[0]
print(f'Загружен: {ZIP_PATH}')

In [ ]:
# ── Загрузка датасета с усреднением ──────────────────────────────────────────
#
# Логика: группируем CSV по (label, iteration) → усредняем все ионы итерации
# → один 200-dim вектор на итерацию. Точно так же работает inference.
#
import zipfile
import io
import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import PurePosixPath

def load_zip_averaged(zip_path: str):
    """
    Читает zip с KDE-массивами, усредняет ионы внутри каждой итерации.
    Возвращает X (n_samples, 200) и y (n_samples,).
    """
    # {(label, iteration): [ion_array, ...]}
    samples = defaultdict(list)

    with zipfile.ZipFile(zip_path, 'r') as zf:
        csv_files = [n for n in zf.namelist() if n.endswith('.csv')]
        print(f'CSV файлов в архиве: {len(csv_files)}')

        for entry in csv_files:
            parts = PurePosixPath(entry).parts
            # Ожидаем: [root?/] label / iteration / ion.csv
            if len(parts) < 3:
                continue
            label     = parts[-3]   # название вещества
            iteration = parts[-2]   # номер итерации

            # Пропускаем служебные файлы (labels.csv и т.п.)
            if not iteration.isdigit():
                continue

            try:
                with zf.open(entry) as f:
                    df = pd.read_csv(io.TextIOWrapper(f, encoding='utf-8'))
                if 'kde_values' in df.columns:
                    arr = df['kde_values'].values.astype(np.float32)
                else:
                    arr = df.iloc[:, 0].values.astype(np.float32)
                if len(arr) != 200:
                    continue
                samples[(label, iteration)].append(arr)
            except Exception as e:
                print(f'  Пропуск {entry}: {e}')
                continue

    if not samples:
        raise ValueError('Нет данных. Проверьте структуру zip: label/iteration/ion.csv')

    X, y = [], []
    label_counts = defaultdict(int)

    for (label, _iter), ion_arrays in samples.items():
        averaged = np.mean(ion_arrays, axis=0)   # ← усредняем ионы
        X.append(averaged)
        y.append(label)
        label_counts[label] += 1

    print('\nРаспределение по классам:')
    for lbl, cnt in sorted(label_counts.items()):
        print(f'  {lbl:30s}  {cnt} сэмплов')

    return np.array(X, dtype=np.float32), np.array(y)


X, y = load_zip_averaged(ZIP_PATH)
print(f'\nX.shape = {X.shape}   (ожидается (n, 200))')
print(f'Классов: {len(set(y))}')

In [ ]:
# ── Train / Test split ────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train: {len(X_train)}  |  Test: {len(X_test)}')

## 1. Random Forest v2

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import seaborn as sns
import matplotlib.pyplot as plt

print('=== Random Forest v2 ===')
rf_v2 = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    n_jobs=-1,
    random_state=42,
)
rf_v2.fit(X_train, y_train)

y_pred_rf = rf_v2.predict(X_test)
print('\nClassification Report:')
print(classification_report(y_test, y_pred_rf))

# 5-fold CV
cv_scores = cross_val_score(rf_v2, X, y, cv=5, scoring='accuracy')
print(f'5-fold CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

# Матрица ошибок
classes = sorted(set(y))
cm = confusion_matrix(y_test, y_pred_rf, labels=classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title('RF v2 — Confusion Matrix')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

joblib.dump(rf_v2, 'rf_optimized_model_v2.pkl')
print('\nСохранено: rf_optimized_model_v2.pkl')

## 2. CatBoost Substance v2

In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report

print('=== CatBoost Substance v2 ===')
cb_v2 = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.1,
    loss_function='MultiClass',
    eval_metric='Accuracy',
    random_seed=42,
    verbose=100,
)
cb_v2.fit(X_train, y_train, eval_set=(X_test, y_test))

y_pred_cb = cb_v2.predict(X_test).flatten()
print('\nClassification Report:')
print(classification_report(y_test, y_pred_cb))

# Матрица ошибок
classes = sorted(set(y))
cm = confusion_matrix(y_test, y_pred_cb, labels=classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=classes, yticklabels=classes)
plt.title('CatBoost Substance v2 — Confusion Matrix')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

cb_v2.save_model('catboost_substance_v2.cbm')
print('\nСохранено: catboost_substance_v2.cbm')

## 3. AutoML ExtraTrees v2 (FLAML)

In [ ]:
from flaml import AutoML
from sklearn.metrics import classification_report
import joblib

print('=== AutoML ExtraTrees v2 (FLAML) ===')
automl = AutoML()
automl.fit(
    X_train, y_train,
    task='classification',
    time_budget=300,           # 5 минут, можно увеличить
    estimator_list=['extra_tree'],
    metric='accuracy',
    seed=42,
    verbose=1,
)

y_pred_aml = automl.predict(X_test)
print('\nClassification Report:')
print(classification_report(y_test, y_pred_aml))

# Матрица ошибок
classes = sorted(set(y))
cm = confusion_matrix(y_test, y_pred_aml, labels=classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=classes, yticklabels=classes)
plt.title('AutoML ExtraTrees v2 — Confusion Matrix')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

# Сохраняем сам sklearn-estimator (совместимо с _load_sklearn_model)
et_v2 = automl.model.estimator
joblib.dump(et_v2, 'extra_trees_v2.pkl')
print(f'\nЛучшие параметры: {automl.best_config}')
print('Сохранено: extra_trees_v2.pkl')

## Скачать все модели

In [ ]:
from google.colab import files
files.download('rf_optimized_model_v2.pkl')
files.download('catboost_substance_v2.cbm')
files.download('extra_trees_v2.pkl')
print('Готово!')